In [18]:
import pandas as pd 

In [3]:
import io
from pathlib import Path

import numpy as np
import pandas as pd


COLONNES_EXACTES = [
    "Temps (s)",
    "Altitude (m)",
    "Vitesse totale (m/s)",
    "Masse (g)",
    "Emplacement du CP (cm)",
    "Emplacement du CG (cm)",
    "Calibres marge de stabilité (​)",
    "Coefficient de force normale (​)",
    "Température de l'air (°C)",
    "Pression atmosphérique (mbar)",
    "Air density (g/cm³)",
    "Mach number (​)",
]


def charger_csv_propre(fichier):
    fichier = Path(fichier)

    with open(fichier, "r", encoding="utf-8", errors="replace") as f:
        lignes = f.readlines()

    if not lignes:
        raise ValueError("Le fichier est vide.")

    header_index = None

    for i, ligne in enumerate(lignes):
        ligne_propre = ligne.strip()

        if not ligne_propre:
            continue

        candidate = ligne_propre.lstrip("#").strip()

        correspondances = sum(col in candidate for col in COLONNES_EXACTES)

        if correspondances >= 3:
            header_index = i
            break

    if header_index is None:
        raise ValueError("Impossible de trouver la vraie ligne d'en-tête du CSV.")

    header = lignes[header_index].lstrip("#").strip()
    separateur = ";" if header.count(";") >= header.count(",") else ","

    lignes_utiles = [header + "\n"]

    for ligne in lignes[header_index + 1:]:
        ligne_propre = ligne.strip()

        if not ligne_propre:
            continue

        if ligne_propre.startswith("#"):
            continue

        lignes_utiles.append(ligne_propre + "\n")

    df = pd.read_csv(
        io.StringIO("".join(lignes_utiles)),
        sep=separateur,
        skipinitialspace=True,
        engine="python",
    )

    df.columns = [str(col).strip().replace('"', "") for col in df.columns]

    return df


def convertir_colonnes_numeriques(df):
    df = df.copy()

    for col in df.columns:
        valeurs = df[col].astype(str).str.strip().str.replace(",", ".", regex=False)
        df[col] = pd.to_numeric(valeurs, errors="coerce")

    df = df.replace([np.inf, -np.inf], np.nan)

    return df


def construire_tableau_projet(df):
    colonnes_presentes = [col for col in COLONNES_EXACTES if col in df.columns]

    if not colonnes_presentes:
        raise ValueError("Aucune des colonnes exactes attendues n'a été trouvée.")

    tableau = df[colonnes_presentes].copy()

    return tableau


if __name__ == "__main__":
    fichier = "open_rocket_data.csv"

    df_brut = charger_csv_propre(fichier)
    df_converti = convertir_colonnes_numeriques(df_brut)
    tableau_projet = construire_tableau_projet(df_converti)

    print("\nColonnes détectées dans le fichier :")
    print(df_brut.columns.tolist())

    print("\nColonnes gardées pour le projet :")
    print(tableau_projet.columns.tolist())

    print("\nAperçu des premières lignes du tableau exploitable :")
    print(tableau_projet.head(20))


Colonnes détectées dans le fichier :
['Temps (s)', 'Altitude (m)', 'Vitesse totale (m/s)', 'Masse (g)', 'Emplacement du CP (cm)', 'Emplacement du CG (cm)', 'Calibres marge de stabilité (\u200b)', 'Coefficient de force normale (\u200b)', "Température de l'air (°C)", 'Pression atmosphérique (mbar)', 'Air density (g/cm³)', 'Mach number (\u200b)']

Colonnes gardées pour le projet :
['Temps (s)', 'Altitude (m)', 'Vitesse totale (m/s)', 'Masse (g)', 'Emplacement du CP (cm)', 'Emplacement du CG (cm)', 'Calibres marge de stabilité (\u200b)', 'Coefficient de force normale (\u200b)', "Température de l'air (°C)", 'Pression atmosphérique (mbar)', 'Air density (g/cm³)', 'Mach number (\u200b)']

Aperçu des premières lignes du tableau exploitable :
    Temps (s)  Altitude (m)  Vitesse totale (m/s)  Masse (g)  \
0        0.00      0.000000                 0.000    23670.0   
1        0.01      0.000000                 0.010    23670.0   
2        0.02      0.000523                 0.136    23660.0   

In [4]:
df_brut

,Temps (s),Altitude (m),Vitesse totale (m/s),Masse (g),Emplacement du CP (cm),Emplacement du CG (cm),Calibres marge de stabilité (​),Coefficient de force normale (​),Température de l'air (°C),Pression atmosphérique (mbar),Air density (g/cm³),Mach number (​)
0,0.000,0.000000,0.000,23670.0,NaN,199.924,NaN,NaN,15.000,1013.250,0.001,0.007
1,0.010,0.000000,0.010,23670.0,NaN,199.921,NaN,NaN,15.000,1013.250,0.001,0.007
2,0.020,0.000523,0.136,23660.0,NaN,199.917,NaN,NaN,15.000,1013.250,0.001,0.007
3,0.030,0.003000,0.407,23660.0,NaN,199.914,NaN,NaN,15.000,1013.250,0.001,0.007
4,0.040,0.009000,0.824,23660.0,NaN,199.911,NaN,NaN,15.000,1013.249,0.001,0.008
...,...,...,...,...,...,...,...,...,...,...,...,...
3367,783.888,5.981000,5.196,15090.0,0.0,188.788,-12.586,NaN,14.961,1012.549,0.001,0.014
3368,784.135,4.781000,5.221,15090.0,0.0,188.788,-12.586,NaN,14.969,1012.689,0.001,0.014
3369,784.383,3.582000,5.233,15090.0,0.0,188.788,-12.586,NaN,14.977,1012.830,0.001,0.014
3370,784.630,2.382000,5.261,15090.0,0.0,188.788,-12.586,NaN,14.985,1012.971,0.001,0.014


In [5]:
df_converti

,Temps (s),Altitude (m),Vitesse totale (m/s),Masse (g),Emplacement du CP (cm),Emplacement du CG (cm),Calibres marge de stabilité (​),Coefficient de force normale (​),Température de l'air (°C),Pression atmosphérique (mbar),Air density (g/cm³),Mach number (​)
0,0.000,0.000000,0.000,23670.0,NaN,199.924,NaN,NaN,15.000,1013.250,0.001,0.007
1,0.010,0.000000,0.010,23670.0,NaN,199.921,NaN,NaN,15.000,1013.250,0.001,0.007
2,0.020,0.000523,0.136,23660.0,NaN,199.917,NaN,NaN,15.000,1013.250,0.001,0.007
3,0.030,0.003000,0.407,23660.0,NaN,199.914,NaN,NaN,15.000,1013.250,0.001,0.007
4,0.040,0.009000,0.824,23660.0,NaN,199.911,NaN,NaN,15.000,1013.249,0.001,0.008
...,...,...,...,...,...,...,...,...,...,...,...,...
3367,783.888,5.981000,5.196,15090.0,0.0,188.788,-12.586,NaN,14.961,1012.549,0.001,0.014
3368,784.135,4.781000,5.221,15090.0,0.0,188.788,-12.586,NaN,14.969,1012.689,0.001,0.014
3369,784.383,3.582000,5.233,15090.0,0.0,188.788,-12.586,NaN,14.977,1012.830,0.001,0.014
3370,784.630,2.382000,5.261,15090.0,0.0,188.788,-12.586,NaN,14.985,1012.971,0.001,0.014


In [6]:
tableau_projet 

,Temps (s),Altitude (m),Vitesse totale (m/s),Masse (g),Emplacement du CP (cm),Emplacement du CG (cm),Calibres marge de stabilité (​),Coefficient de force normale (​),Température de l'air (°C),Pression atmosphérique (mbar),Air density (g/cm³),Mach number (​)
0,0.000,0.000000,0.000,23670.0,NaN,199.924,NaN,NaN,15.000,1013.250,0.001,0.007
1,0.010,0.000000,0.010,23670.0,NaN,199.921,NaN,NaN,15.000,1013.250,0.001,0.007
2,0.020,0.000523,0.136,23660.0,NaN,199.917,NaN,NaN,15.000,1013.250,0.001,0.007
3,0.030,0.003000,0.407,23660.0,NaN,199.914,NaN,NaN,15.000,1013.250,0.001,0.007
4,0.040,0.009000,0.824,23660.0,NaN,199.911,NaN,NaN,15.000,1013.249,0.001,0.008
...,...,...,...,...,...,...,...,...,...,...,...,...
3367,783.888,5.981000,5.196,15090.0,0.0,188.788,-12.586,NaN,14.961,1012.549,0.001,0.014
3368,784.135,4.781000,5.221,15090.0,0.0,188.788,-12.586,NaN,14.969,1012.689,0.001,0.014
3369,784.383,3.582000,5.233,15090.0,0.0,188.788,-12.586,NaN,14.977,1012.830,0.001,0.014
3370,784.630,2.382000,5.261,15090.0,0.0,188.788,-12.586,NaN,14.985,1012.971,0.001,0.014


In [9]:
tableau_projet.isna()

,Temps (s),Altitude (m),Vitesse totale (m/s),Masse (g),Emplacement du CP (cm),Emplacement du CG (cm),Calibres marge de stabilité (​),Coefficient de force normale (​),Température de l'air (°C),Pression atmosphérique (mbar),Air density (g/cm³),Mach number (​)
0,False,False,False,False,True,False,True,True,False,False,False,False
1,False,False,False,False,True,False,True,True,False,False,False,False
2,False,False,False,False,True,False,True,True,False,False,False,False
3,False,False,False,False,True,False,True,True,False,False,False,False
4,False,False,False,False,True,False,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
3367,False,False,False,False,False,False,False,True,False,False,False,False
3368,False,False,False,False,False,False,False,True,False,False,False,False
3369,False,False,False,False,False,False,False,True,False,False,False,False
3370,False,False,False,False,False,False,False,True,False,False,False,False
